# Olist e-commerce - exploratory data analysis
Source: DuckDB star schema `FDL_DATA` (built by `pipeline/run_all.py`). All queries live in `analysis/eda.py`
so the notebook, the Streamlit app and the slide findings share one implementation.
*Valid orders = status not in (canceled, unavailable). Revenue = sum of item price; GMV = price + freight.*

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd, plotly.express as px
from analysis import eda
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.width = 160

## 0. Data model sanity check - row counts of the star schema

In [2]:
eda.query('''SELECT table_name, estimated_size AS rows FROM duckdb_tables()
             WHERE schema_name = 'FDL_DATA' ORDER BY table_name''')

,table_name,rows
0,DQ_RESULT_LOG,501
1,D_CUSTOMER,96097
2,D_DATE,1828
3,D_GEOGRAPHY,19178
4,D_PAYMENT_TYPE,6
5,D_PRODUCT,32952
6,D_SELLER,3096
7,ETL_RUN_LOG,30
8,F_ORDERS,99441
9,F_ORDER_ITEMS,112650


## 1. Headline KPIs

In [3]:
eda.kpis().to_frame('value')

,value
orders,"98,199.00"
items_sold,"112,101.00"
revenue,"13,494,400.74"
gmv_incl_freight,"15,735,527.03"
avg_order_value,137.42
customers,"94,983.00"
sellers,"3,053.00"
products,"32,729.00"


## 2. Monthly sales trends
Months with fewer than 100 orders (Sep-2016, Dec-2016, Sep-2018) are data-edge artefacts and are excluded from trend statements.

In [4]:
m = eda.monthly_sales()
m

,year_month,month_start,orders,items,revenue,gmv,avg_order_value,revenue_mom_pct,revenue_3m_avg
0,2016-09,2016-09-04,2,5,207.86,279.69,103.93,NaN,NaN
1,2016-10,2016-10-03,290,342,"44,507.30","51,354.52",153.47,"21,312.20",NaN
2,2016-12,2016-12-23,1,1,10.90,19.62,10.90,-100.00,"14,908.69"
3,2017-01,2017-01-05,787,953,"120,098.27","136,943.46",152.60,"1,101,719.00","54,872.16"
4,2017-02,2017-02-01,1718,1936,"244,959.35","283,561.69",142.58,104.00,"121,689.51"
5,2017-03,2017-03-01,2617,2975,"368,341.32","425,617.96",140.75,50.40,"244,466.31"
6,2017-04,2017-04-01,2377,2660,"353,842.98","405,848.61",148.86,-3.90,"322,381.22"
7,2017-05,2017-05-01,3640,4106,"503,159.19","582,710.83",138.23,42.20,"408,447.83"
8,2017-06,2017-06-01,3205,3571,"429,916.61","499,652.24",134.14,-14.60,"428,972.93"
9,2017-07,2017-07-01,3946,4493,"492,287.30","578,753.73",124.76,14.50,"475,121.03"


In [5]:
full = m[m.orders >= 100]
fig = px.bar(full, x='year_month', y='revenue', title='Monthly revenue')
fig.add_scatter(x=full.year_month, y=full.revenue_3m_avg, name='3-month average')
fig.show()

In [6]:
y17 = m[m.year_month.between('2017-01','2017-08')].revenue.sum(); y18 = m[m.year_month.between('2018-01','2018-08')].revenue.sum()
print(f'Jan-Aug revenue: 2017 = {y17:,.0f}  2018 = {y18:,.0f}  growth = {(y18/y17-1)*100:+.1f}%')
print('Peak month:', full.loc[full.revenue.idxmax(), 'year_month'])

Jan-Aug revenue: 2017 = 3,080,851  2018 = 7,341,037  growth = +138.3%
Peak month: 2017-11


## 3. Top-selling products and categories

In [7]:
eda.top_categories(10)

,category,units,orders,revenue,avg_price,revenue_share_pct,cum_share_pct
0,health_beauty,9634,8800,"1,255,695.13",130.34,9.31,9.31
1,watches_gifts,5970,5604,"1,198,185.21",200.70,8.88,18.19
2,bed_bath_table,11097,9399,"1,035,964.06",93.36,7.68,25.87
3,sports_leisure,8590,7673,"979,740.92",114.06,7.26,33.13
4,computers_accessories,7781,6654,"904,322.02",116.22,6.70,39.83
5,furniture_decor,8298,6425,"727,465.05",87.67,5.39,45.22
6,housewares,6915,5847,"626,825.80",90.65,4.65,49.87
7,cool_stuff,3779,3616,"620,770.49",164.27,4.60,54.47
8,auto,4204,3872,"586,585.73",139.53,4.35,58.82
9,garden_tools,4328,3505,"481,009.94",111.14,3.56,62.38


In [8]:
eda.top_products(10, 'revenue')

,product_id,category,units,revenue,avg_price
0,bb50f2e236e5eea0100680137654686c,health_beauty,195,"63,885.00",327.62
1,6cdd53843498f92890544667809f1595,health_beauty,156,"54,730.20",350.83
2,d6160fb7873f184099d9bc95e30376af,computers,35,"48,899.34","1,397.12"
3,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,341,"46,916.51",137.59
4,99a4788cb24856965c36a24e339b6058,bed_bath_table,487,"42,938.66",88.17
5,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,"41,082.60",149.94
6,25c38557cf793876c5abdd5931f922db,baby,38,"38,907.32","1,023.88"
7,5f504b3a1c75b73d6151be81eb05bdc9,cool_stuff,63,"37,733.90",598.95
8,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,"37,683.42",116.67
9,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,"37,608.90",71.36


In [9]:
eda.top_products(10, 'units')

,product_id,category,units,revenue,avg_price
0,aca2eb7d00ea1a7b8ebd4e68314663af,furniture_decor,527,"37,608.90",71.36
1,99a4788cb24856965c36a24e339b6058,bed_bath_table,487,"42,938.66",88.17
2,422879e10f46682990de24d770e7f83d,garden_tools,484,"26,577.22",54.91
3,389d119b48cf3043d311335e499d9c6b,garden_tools,391,"21,390.69",54.71
4,368c6c730842d78016ad823897a372db,garden_tools,388,"21,056.80",54.27
5,53759a2ecddad2bb87a079a1f1519f73,garden_tools,373,"20,387.20",54.66
6,d1c427060a0f73f6b889a5c7c61f2ac4,computers_accessories,341,"46,916.51",137.59
7,53b36df67ebb7c41585e8d54d6772e08,watches_gifts,323,"37,683.42",116.67
8,154e7e31ebfa092203795c972e5804a6,health_beauty,281,"6,325.19",22.51
9,3dd2a17168ec895c781a9191c1e95ad7,computers_accessories,274,"41,082.60",149.94


In [10]:
s = eda.seller_concentration()
print(f"Top 10% of sellers ({int(len(s)*.1)}) = {s.iloc[int(len(s)*.1)-1].cum_share_pct}% of revenue")

Top 10% of sellers (305) = 67.43% of revenue


## 4. Customer segmentation by purchase behaviour
Olist `customer_id` is per order; the real customer is `customer_unique_id` (= one row of `D_CUSTOMER`).

In [11]:
rfm = eda.customer_rfm()
print('Repeat-purchase rate (%):', eda.repeat_rate(rfm))
eda.segment_summary(rfm)

Repeat-purchase rate (%): 3.04


,segment,customers,revenue,avg_clv,avg_orders,avg_recency_days,customer_share_pct,revenue_share_pct
0,Big-ticket one-time,16375,"6,378,495.93",389.53,1.00,305.23,17.24,40.54
1,Recent one-time,34959,"3,131,438.79",89.57,1.00,118.18,36.80,19.90
2,Lapsed one-time,35212,"3,111,505.66",88.36,1.00,371.08,37.07,19.77
3,New high-value,5556,"2,223,828.15",400.26,1.00,64.48,5.85,14.13
4,Loyal repeat,1584,"497,702.96",314.21,2.14,118.30,1.67,3.16
5,At-risk repeat,1304,"392,555.54",301.04,2.08,359.88,1.37,2.49


In [12]:
rfm.groupby('frequency').agg(customers=('customer_key','count'), avg_clv=('monetary','mean')).round(2)

,customers,avg_clv
frequency,,
1,92102,161.18
2,2652,291.08
3,188,430.50
4,29,772.64
5,9,734.18
6,5,691.11
7,3,946.85
9,1,"1,172.67"
16,1,902.04


In [13]:
px.scatter(rfm.sample(5000, random_state=1), x='recency_days', y='monetary', color='segment',
           log_y=True, opacity=.5, title='Recency vs lifetime value (5,000 sampled customers)').show()

## 5. Delivery performance vs. customer reviews

In [14]:
eda.review_vs_lateness()

,delivery,reviews,avg_score,negative_pct
0,Late,6409,2.27,62.40
1,On time,89944,4.29,9.30


In [15]:
eda.review_by_delay_bucket()

,delivery_vs_promise,reviews,avg_score
0,1) 7+ days early,76175,4.31
1,2) 0-6 days early / on day,13769,4.16
2,3) 1-7 days late,3612,2.71
3,4) 8+ days late,2797,1.70


In [16]:
eda.delivery_by_state().head(10)

,state,region,delivered_orders,avg_delivery_days,avg_estimated_days,late_pct
0,RR,North,41,29.40,46.00,12.20
1,AP,North,67,27.20,46.20,3.00
2,AM,North,145,26.40,45.30,2.80
3,AL,Northeast,397,24.50,32.60,21.40
4,PA,North,946,23.80,37.20,11.20
5,MA,Northeast,717,21.60,30.50,17.40
6,SE,Northeast,335,21.50,30.80,15.20
7,CE,Northeast,1279,21.30,31.40,13.80
8,AC,North,80,21.00,41.10,3.80
9,PB,Northeast,517,20.40,33.00,10.40


## 6. Payments and geography

In [17]:
eda.payment_mix()

,payment_type,payments,value,avg_installments,value_share_pct
0,Credit Card,76795,"12,542,084.19",3.51,78.34
1,Boleto,19784,"2,869,361.27",1.00,17.92
2,Voucher,5775,"379,436.87",1.00,2.37
3,Debit Card,1529,"217,989.79",1.00,1.36
4,Not Defined,3,0.00,1.00,0.00


In [18]:
eda.revenue_by_state().head(10)

,state,region,orders,revenue,customers,revenue_share_pct
0,SP,Southeast,41125,"5,163,867.22",39747,38.27
1,RJ,Southeast,12697,"1,811,623.42",12241,13.43
2,MG,Southeast,11496,"1,573,508.20",11134,11.66
3,RS,South,5415,"742,559.78",5233,5.50
4,PR,South,4982,"676,883.06",4824,5.02
5,SC,South,3599,"518,578.28",3501,3.84
6,BA,Northeast,3344,"507,108.83",3244,3.76
7,DF,Central-West,2120,"300,886.45",2057,2.23
8,GO,Central-West,1998,"287,870.46",1934,2.13
9,ES,Southeast,2018,"273,532.13",1950,2.03


## 7. Takeaways
* Revenue grew strongly through 2017 and plateaued between BRL 0.84M and 0.99M/month in 2018 (see section 2).
* Demand is broad across categories (no category > 10% of revenue) but concentrated among sellers (top 10% ~ two thirds of revenue).
* ~97% of customers buy once - retention, not acquisition mix, is the largest commercial lever.
* Late deliveries cut the average review score roughly in half; remote Northern/Northeastern states wait 2-3x longer than Sao Paulo.